# 🧪 DAM-CMA: Multimodal Fake News Detection Dashboard
### Domain Adversarial Learning & Cross-Modal Attention

This notebook serves as the unified interface for the research project. It handles data loading, model training, baseline comparison, and explainability visualization.

In [1]:
import os
import torch
import pandas as pd
import matplotlib.pyplot as plt
from src.data_loader import get_dataloader
from src.models.dam_cma import DAM_CMA_Model
from baselines.simple_fusion import SimpleFusionModel
from baselines.unimodal import TextOnlyBaseline, ImageOnlyBaseline
from src.train_utils import train_model
from src.utils.visualizer import generate_attention_heatmaps

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

ModuleNotFoundError: No module named 'matplotlib'

## 1. Data Statistics
Visualize the composition of the unified dataset.

In [ ]:
df = pd.read_json("data/master_data.jsonl", lines=True)
print(f"Total items in master dataset: {len(df)}")

plt.figure(figsize=(10, 4))
plt.subplot(1, 2, 1)
df['label'].value_counts().plot(kind='bar', color=['red', 'green'])
plt.title("Label Distribution (1=Fake, 0=Real)")

plt.subplot(1, 2, 2)
df['source'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title("Source Distribution")
plt.show()

## 2. Model Training Configuration
Initialize the main DAM-CMA model and comparative baselines.

In [ ]:
# Prepare Data Loaders
train_loader = get_dataloader("data/master_data.jsonl", batch_size=16, shuffle=True)

# 1. Main Research Model (DAM-CMA)
main_model = DAM_CMA_Model(num_domains=2).to(device)

# 2. Baselines
fusion_baseline = SimpleFusionModel().to(device)
text_only = TextOnlyBaseline().to(device)
image_only = ImageOnlyBaseline().to(device)

## 3. Comparative Analysis Execution
Train all models and compare results.

In [ ]:
print("Training DAM-CMA (Adversarial Mode)...")
train_model(main_model, train_loader, epochs=5, is_adversarial=True, device=device)
torch.save(main_model.state_dict(), "models/dam_cma_final.pth")

print("\nTraining Baselines...")
train_model(fusion_baseline, train_loader, epochs=3, device=device)
train_model(text_only, train_loader, epochs=3, device=device)
train_model(image_only, train_loader, epochs=3, device=device)

print("\n✅ Research Training Phase Complete!")

## 4. Explainability: Cross-Modal Attention Heatmaps
Generate heatmaps to show where the model is looking (Image regions vs. Keywords).

In [ ]:
# Select a few samples for visualization
test_samples = df.sample(3)

for idx, sample in test_samples.iterrows():
    print(f"ID: {sample['id']} | True Label: {sample['label']}")
    print(f"Text: {sample['text']}")
    generate_attention_heatmaps(main_model, sample['text'], sample['image_path'], device=device)
    plt.show()